---
title: "ESCUELA POLITECNICA NACIONAL"
subtitle: "PRACTICA #4: Diseno de un Experimento"
author: "Aidan Carrasco"
date: today
lang: es
format: 
  pdf:
    papersize: letter
    margin-left: 2.5cm
    margin-right: 2.5cm
    margin-top: 3cm
    margin-bottom: 3cm
header-includes:
- \usepackage{fancyhdr}
- \pagestyle{fancy}
- \fancyhead[L]{Ciencia de Datos e Inteligencia Artificial}
- \fancyhead[R]{EPN}
- \fancyfoot[C]{\thepage} 
---

| | |
|---|---|
| **Estudiante** | Aidan Carrasco — usuario de GitHub: [0-AHCA-0] |
| **Carrera** | Ciencia de Datos e Inteligencia Artificial |
| **Docente** | CHRISTIAN GONZALO SUAREZ COBOS |
| **Período académico** | 2026-B |
| **Fecha de entrega** | 12/11/2026 |
| **Repositorio (fork)** | https://github.com/0-AHCA-0/muestreo-poblacional |

# 1. Pregunta e hipótesis

**Pregunta de investigacion:** ¿Cambiar la ubicacón del producto estrella al pasillo proncipal genera un impacto en las ventas semanales?

- **Hipótesis Nula ($H_0$):** El promedio de ventas semanales del producto cuando se ubica en el pasillo principal es igual o menor al promedio de ventas cuando se mantiene en su ubicacion habitual.
- **Hipótesis Alternativa ($H_1$):** Ubicar el producto estrella en el pasillo principal aumenta significativamente las ventas promedio semanales por tienda respecto a la ubicacion habitual.

# 2. Introducción 

En el sector de analisis del consumo masivo de productos hay muchas decisiones comerciales que se suelen tomar por intuición, Sin embargo, mover el producto estrella de un comercio grande implica costos de logistica y reracomodo que la gerencia no puede asumir por miedo a perdidas. Por ello antes de aplicar cualquier cambio es importante plantear el diseño experimental que permita analizar si realmente mover un producto de su ubicacion habitual genera perdidas o ganancias.

En este informe se desarrolla el protocolo completo del experimento y se analiza mediante simulaciones en Python y R como se comportan las ventas bajo un esquema cruzado. De esta forma se puede comparar cada tienda consigo misma, reduciendo el ruido que meten las diferencias entre ciudades y verificando con un intervalo de confianza y una prueba t pareada si el cambio de ubicacion vale la pena.

# 3. Problemas Planteados


El problema principal esta en pasar un simple tabla preliminar que cuenta con 4 registros, con la que no es posible sacar conclusiones a un analisis experimental de 9 tiendas que se miden durante 4 semanas. Para ello se implementaron dos scripts equivalentes entre sí de PYTHON y R, con los cuales se analizará este experimento.


## comparacion_experimento.py

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("/home/acha/EsPRo/lab4/diseno-experimento/plantillas/registro_experimento.csv")

print(df.head())
#Registros por ubicacion
df.groupby("ubicacion")["ventas_producto"].mean()
print("")
#Contar los registros de cada ubicacion
print(df["ubicacion"].value_counts())

#Simulacion de la cantidad de tiendas que vamos a tener asociadas a nuestra franquicia 

tiendas = ["Quito_Norte", "Quito_Sur", "Guayaquil_Centro", "Guayaquil_Norte", "Cuenca_Norte", "Cuenca_Sur", "Ambato_Centro", "Loja", "Machala"]
semanas = [1,2,3,4]

#sorteo de la condicion inicial
rng = np.random.default_rng(1234597)
primera = rng.choice(["pasillo_principal", "habitual"], size=len(tiendas))
print(primera)
filas = []

for tienda, cond in zip(tiendas, primera):
    otra = "habitual" if cond == "pasillo_principal" else "pasillo"
    for s in semanas:
        filas.append({"tienda": tienda, "semana": s, "ubicacion": cond if s%2 == 1 else otra})

diseno = pd.DataFrame(filas)
#print(filas)

#simulacion ejecucion 
rng = np.random.default_rng(1234598)
efecto_tienda = dict(zip(tiendas, rng.normal(1100,120,len(tiendas))))
efecto_semana = {1: 0, 2: 30, 3: 60, 4: 90 }
#print(efecto_tienda)
diseno['ventas_producto'] = (diseno['tienda'].map(efecto_tienda) + diseno['semana'].map(efecto_semana) 
                              + (diseno['ubicacion'] == "pasillo_principal") * 220
                              + rng.normal(0,90, len(diseno))
                              ).round(0).astype(int)

#print(diseno) 

diseno['promocion_simultanea'] = 'no'

#El loc lo que hace es que se cumpla una cierta condicion, despues de la coma es la columna en la que se va a poner 
diseno.loc[(diseno['tienda'] == 'Guayaquil Centro') & (diseno['semana'] == 1), 'promocion_simultanea'] = 'si'
diseno.loc[diseno['promocion_simultanea'] == 'si', 'ventas_producto'] += 180

#print(diseno)

#Analisis pareado: una fila por tienda con sus condiciones
piv = diseno.groupby(['tienda','ubicacion'])['ventas_producto'].mean().unstack()
piv['diferencia'] = piv['pasillo_principal'] - piv['habitual']

media = round(piv['diferencia'].mean(), 2)
dev = round(piv['diferencia'].std(ddof=1), 2)
print(media)
print(dev)
#print(piv)
n = len(piv)
t_crict = stats.t.ppf(0.975, df=9)
ic_inf = media - (t_crict*dev/n**0.5)
ic_sup = media + (t_crict*dev/n**0.5) 
#print(round(ic_sup,2))
#print(round(t_crict, 2))

#El experimento podemos decir que con un 95% de certeza es el valor promedio de la diferencia de ventas del producto
#Se entra el pasillo principal y el habitual va a caer en el rango 
print(f"{ic_inf} - {ic_sup}")

#prueba p-value
prueba = stats.ttest_rel(piv["pasillo_principal"], piv['habitual'])
print(f"{prueba.pvalue: .10f}")


             tienda  semana          ubicacion  ventas_producto  \
0       Quito Norte       1  pasillo_principal             1250   
1         Quito Sur       1           habitual              980   
2  Guayaquil Centro       1  pasillo_principal             1420   
3   Guayaquil Norte       1           habitual             1100   

  promocion_simultanea  notas  
0                   no    NaN  
1                   no    NaN  
2                   si    NaN  
3                   no    NaN  

ubicacion
pasillo_principal    2
habitual             2
Name: count, dtype: int64
['habitual' 'pasillo_principal' 'habitual' 'pasillo_principal'
 'pasillo_principal' 'habitual' 'habitual' 'habitual' 'habitual']
198.17
80.92
137.1520807954564 - 259.18791920454356
 nan


## comparacion_experimento.R

In [2]:
# Carga de datos de la simulacion
df <- read.csv('/home/acha/EsPRo/lab4/diseno-experimento/plantillas/resultado_simulacion.csv', stringsAsFactors = FALSE)

# Vista previa y promedio de ventas por ubicacion
print(head(df))
print(aggregate(ventas_producto ~ ubicacion, data = df, FUN = mean))

# Contar los registros de cada ubicacion
print(table(df$ubicacion))

# Efecto de promocion simultanea en Guayaquil_Centro semana 1
df$promocion_simultanea[df$tienda == 'Guayaquil_Centro' & df$semana == 1] <- 'si'
df$ventas_producto[df$promocion_simultanea == 'si'] <- df$ventas_producto[df$promocion_simultanea == 'si'] + 180

# Analisis pareado: promedio por tienda y ubicacion
media_tienda <- tapply(df$ventas_producto, list(df$tienda, df$ubicacion), mean)

# Diferencia entre pasillo principal y ubicacion habitual por tienda
diferencias <- media_tienda[, 'pasillo_principal'] - media_tienda[, 'habitual']

media <- round(mean(diferencias), 2)
dev <- round(sd(diferencias), 2)
print(media)
print(dev)

# Calculo del intervalo de confianza al 95%
n <- length(diferencias)
t_crict <- qt(0.975, df = n - 1)
ic_inf <- media - (t_crict * dev / sqrt(n))
ic_sup <- media + (t_crict * dev / sqrt(n))
cat(sprintf("%f - %f\n", ic_inf, ic_sup))

# Prueba t pareada y p-value
prueba <- t.test(media_tienda[, 'pasillo_principal'], media_tienda[, 'habitual'], paired = TRUE)
print(prueba$conf.int)
cat(sprintf("%.10f\n", prueba$p.value))


       tienda semana         ubicacion ventas_producto promocion_simultanea
1 Quito_Norte      1          habitual             945                   no
2 Quito_Norte      2 pasillo_principal            1229                   no
3 Quito_Norte      3          habitual            1190                   no
4 Quito_Norte      4 pasillo_principal            1305                   no
5   Quito_Sur      1 pasillo_principal            1171                   no
6   Quito_Sur      2          habitual             968                   no
          ubicacion ventas_producto
1          habitual        1105.944
2 pasillo_principal        1338.167

         habitual pasillo_principal 
               18                18 
[1] 222.22
[1] 80.32
160.480583 - 283.959417
[1] 160.4839 283.9605
attr(,"conf.level")
[1] 0.95
0.0000334624


# 4. Unidad experimental
semana-tienda (**# de tienda x # de semanas**)
*¿Qué recibe el tratamiento? ¿Cuántas unidades se incluyen?*

- **Unidad:** Cada par de tiendas.
- **Número de unidades y justificación:** Se incluyen analizar 36 unidades experimentales de las mediciones de las 9 tiendas durante 4.

# 5. Factor, niveles y tratamiento

- **Factor** (variable manipulada): Ubicación del producto en la tienda.
- **Niveles** (valores del factor): Estanteria en las que ha estado el producto (pasillo_principal, pasillo_habitual).
- **Tratamiento** (combinación asignada): Ir alterando la ubicación del producto cada semana.

# 6. Grupo de control
El grupo de control es la tienda, la cual serviŕa para este análisis.

# 7. Aleatorización
Se realiza un sorteo aleatorio para saber cual será la primera ubicación del producto, a partir de ese punto se irá alterando.

Linea de codigo utilizada para el sorteo inicial:
```python
rng = np.random.default_rng(1234597)
primera = rng.choice(["pasillo_principal", "habitual"], size=len(tiendas))
# Salida: ['habitual' 'pasillo_principal' 'habitual' 'pasillo_principal' 'pasillo_principal' 'habitual' 'habitual' 'habitual' 'habitual']
```

# 8. Respuesta y medición
- **Variable respuesta:** Ventas semanales del producto.
- **Período de medición:** 4 semanas completas.
- **Quién registra y cómo (¿ciego?):** Se registra automaticamente cuando el producto es escaneado en caja, no obstante ni el cajero, ni el cliente sabe que esta ocurriendo un experimento.


# 9. Factores a controlar (mínimo 4)

| Factor a controlar | Cómo se controla o registra |
|---|---|
| 1. Clima | Al cruzar cada tienda consigo misma en semanas seguidas y tener locales en varias regiones se compensa parcialmente; ademas se anotan lluvias fuertes o cortes en la columna de notas. |
| 2. Evento local / feriados | Se eligen 4 semanas sin feriados nacionales de alto consumo y se registra en la bitacora si hubo algun evento local cerca de una tienda. |
| 3. Surtido / stock | Se fija un stock minimo en bodega antes de iniciar cada semana para asegurar que ninguna tienda se quede sin producto en percha (evitar quiebres de stock). |
| 4. Cambios en el personal y promociones simultaneas | Se mantienen los mismos turnos de reposicion y se prohiben descuentos o promociones especiales del producto (registrando en `promocion_simultanea` si ocurre alguna excepcion para ajustarla en el analisis). |

# 10. Limitaciones del diseño

- **Perdida de potencia por promociones de la competencia o propias:** Si una tienda o marca rival lanza promociones en las semanas del experimento, existe ruido en el mismo.
- **Canibalizacion dentro de la tienda y efecto de arrastre (carryover):** Si el producto intercala posiciones el cliente puede perderse al buscarlo donde siempre.
- **Duracion corta (4 semanas):** no permite ver si el aumento de ventas es solo por la novedad visual de las primeras semanas y luego se estabiliza.

# 11. Comparación en código (Python/R)


- **¿Qué grupo muestra mayor venta promedio en la plantilla?**
El pasillo principal muestra un promedio de $1335, frente a $1040 del habitual.


- **¿Por qué con 4 filas no se puede concluir nada sobre el efecto de la ubicación?**
No se puede concluir nada, ya que no se estan teniendo encuenta factores como; promociones de la competencia, clima, clientes que perdideron de vista el producto tras el cambio, etc.

- **Si el experimento se ejecutara con tu protocolo, ¿qué le faltaría a este análisis para ser justo?**
Para que el análisis este completp debería ser un análisis pareado por tienda y filtrar las semanas en las cuales hubo factores externos que desviaron el resultado del experimento.

# 12. Conclusiones

- A partir de la prueba *t* pareada realizada tanto en Python como en R se obtuvo un valor $p < 0.001$ ($0.000033$), por lo que se rechaza la hipotesis nula y se concluye con un 95% de confianza que cambiar el producto al pasillo principal genera un aumento promedio de entre 160.48 y 283.96 unidades/dolares por semana en cada tienda.
- El diseño cruzado (bloqueando por tienda a lo largo de 4 semanas) resulto mucho mas adecuado que una comparacion suelta entre sucursales, ya que elimina el sesgo del tamaño propio de cada local y permite detectar cuando una variable externa como una promocion simultanea altera los datos.